# kd-capacity-gap — v2 Experiment Runner (Colab Pro+ / A100)

End-to-end pipeline for the v2 evaluation protocol. **Run cells top-to-bottom.**
Every stage skips completed work (via `results.json`), and a background thread
syncs metrics + teacher checkpoints to Drive every 10 minutes. After a
disconnect: reconnect an A100 runtime and re-run from the top — completed
stages SKIP in seconds and execution resumes where it stopped.

| Stage | What happens | Est. time (A100) |
|-------|-------------|------------------|
| 1a | Teachers: R50, R34, R101 × 3 seeds (200 ep) | done ✓ |
| 1b | Selection grid: 4 pairs × 12 configs, seed 0 (val-based) | done ✓ |
| 1c | `collect_results --write-best` → `best_configs.json` | done ✓ |
| 2 | Finals: baselines + best configs × 5 seeds + fidelity | ~28–34 h |
| 2b | Bug ablation: bugged Feature-KD, R50→R18, 5 seeds | ~2 h |
| 3 | Stem ablation | ~8–10 h |
| 4 | Aggregate + download metrics (small zip) | 1 min |

> Enable **background execution** (Pro+) so runs continue with the browser
> closed. Stage 2 will not fit in one session — expected; resume works.
>
> ⚠ Student checkpoints are **not** synced to Drive (too large). If a session
> dies between a run finishing and its fidelity eval, that run's
> `fidelity.json` will be missing — the backfill cell after Stage 2 detects
> and repairs these.

## 0 — GPU check

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "NO GPU"
# If "NO GPU": Runtime -> Change runtime type -> A100 GPU, then re-run.

NVIDIA A100-SXM4-80GB, 81920 MiB


## 0.5 — Drive mount + background sync

`DRIVE_BASE` controls where outputs persist. The sync loop copies only:
- every `results.json`, `fidelity.json`, `best_configs.json`, `*.csv`
- teacher checkpoints (`checkpoints/teacher_*.pth` and `runs/teachers/**/checkpoint_best.pth`)

Student checkpoints stay local (disposable — fidelity is computed in-session).

In [2]:
from google.colab import drive
import os, time, shutil, threading
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/kd-capacity-gap-v2')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f'Drive base: {DRIVE_BASE}')

REPO = Path('/content/kd-capacity-gap')

SYNC_PATTERNS = ['results.json', 'fidelity.json']

def _iter_sync_files():
    if not REPO.exists():
        return
    for pat in SYNC_PATTERNS:
        yield from (REPO / 'runs').rglob(pat) if (REPO / 'runs').exists() else []
    for extra in ['best_configs.json', 'final_results.csv']:
        p = REPO / extra
        if p.exists():
            yield p
    ck = REPO / 'checkpoints'
    if ck.exists():
        yield from ck.glob('teacher_*.pth')
    tr = REPO / 'runs' / 'teachers'
    if tr.exists():
        yield from tr.rglob('checkpoint_best.pth')

def sync_to_drive(verbose=False):
    n = 0
    for src in _iter_sync_files():
        rel = src.relative_to(REPO)
        dst = DRIVE_BASE / rel
        if not dst.exists() or dst.stat().st_mtime < src.stat().st_mtime:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            n += 1
    if verbose or n:
        print(f'[sync] {n} file(s) → Drive  ({time.strftime("%H:%M:%S")})')

def restore_from_drive():
    n = 0
    for src in DRIVE_BASE.rglob('*'):
        if not src.is_file():
            continue
        dst = REPO / src.relative_to(DRIVE_BASE)
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            n += 1
    print(f'[restore] {n} file(s) ← Drive')

def _sync_loop():
    while True:
        time.sleep(600)  # every 10 min
        try:
            sync_to_drive()
        except Exception as e:
            print(f'[sync] error: {e}')

if not any(t.name == 'drive-sync' for t in threading.enumerate()):
    threading.Thread(target=_sync_loop, name='drive-sync', daemon=True).start()
    print('Background sync started (every 10 min).')

Mounted at /content/drive
Drive base: /content/drive/MyDrive/kd-capacity-gap-v2
Background sync started (every 10 min).


## 1 — Clone repo, restore state, install deps

In [3]:
import subprocess

REPO_URL = 'https://github.com/umutonuryasar/kd-capacity-gap.git'

if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
print('Working directory:', os.getcwd())

restore_from_drive()   # brings back results.json + teacher ckpts -> completed work is skipped


Working directory: /content/kd-capacity-gap
[restore] 167 file(s) ← Drive


In [4]:
# torch/torchvision are pre-installed on Colab
!pip install -r requirements.txt -q
print('Requirements ready.')
# CIFAR-10 downloads automatically to data/ on the first train.py call — no manual prep.

Requirements ready.


In [5]:
# CIFAR-10 archive: Drive cache <-> local
from pathlib import Path
import shutil

Path('data').mkdir(exist_ok=True)
tgz = Path('data/cifar-10-python.tar.gz')
drive_tgz = DRIVE_BASE / 'cifar-10-python.tar.gz'

if not tgz.exists() and drive_tgz.exists():
    shutil.copy2(drive_tgz, tgz)
    print('CIFAR archive restored from Drive.')
elif tgz.exists() and not drive_tgz.exists():
    shutil.copy2(tgz, drive_tgz)
    print('CIFAR archive cached to Drive.')
else:
    print('Archive status — local:', tgz.exists(), '| Drive:', drive_tgz.exists())

CIFAR archive restored from Drive.


In [6]:
# CIFAR-10 fetch — robust version: cleans up failed attempts, tries mirrors in order.
import hashlib, urllib.request, shutil
from pathlib import Path

Path('data').mkdir(exist_ok=True)
tgz = Path('data/cifar-10-python.tar.gz')
GOOD_MD5 = 'c58f30108f718f92721af3b95e74349a'

MIRRORS = [
    'https://ossci-datasets.s3.amazonaws.com/cifar-10-python.tar.gz',
    'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz',
]

def md5_of(p):
    return hashlib.md5(p.read_bytes()).hexdigest()

# Remove any empty/corrupt leftover from previous attempts
if tgz.exists() and md5_of(tgz) != GOOD_MD5:
    print(f'Removing corrupt/partial file ({tgz.stat().st_size/1e6:.1f} MB)')
    tgz.unlink()

if not tgz.exists():
    for url in MIRRORS:
        print(f'Trying {url} ...')
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r, open(tgz, 'wb') as f:
                shutil.copyfileobj(r, f)
            if md5_of(tgz) == GOOD_MD5:
                break
            print(f'  MD5 mismatch from this mirror, trying next.')
            tgz.unlink()
        except Exception as e:
            print(f'  FAILED: {e}')
            if tgz.exists():
                tgz.unlink()

assert tgz.exists() and md5_of(tgz) == GOOD_MD5, 'All mirrors failed — re-run this cell.'
print(f'CIFAR-10 archive ready ({tgz.stat().st_size/1e6:.0f} MB, md5 OK)')

CIFAR-10 archive ready (170 MB, md5 OK)


## 2 — Stage 1a: Teachers (R50, R34, R101 × 3 seeds, 200 ep)

`tools/train_teachers.sh` skips any (arch, seed) whose `results.json` exists.
Canonical checkpoints (`checkpoints/teacher_*.pth`) are the seed-0 best-val weights.

In [7]:
!bash tools/train_teachers.sh
sync_to_drive(verbose=True)

  SKIP resnet50 seed0 — already trained.
  SKIP resnet50 seed1 — already trained.
  SKIP resnet50 seed2 — already trained.
  Saved canonical teacher: checkpoints/teacher_r50.pth
  SKIP resnet34 seed0 — already trained.
  SKIP resnet34 seed1 — already trained.
  SKIP resnet34 seed2 — already trained.
  Saved canonical teacher: checkpoints/teacher_r34.pth
  SKIP resnet101 seed0 — already trained.
  SKIP resnet101 seed1 — already trained.
  SKIP resnet101 seed2 — already trained.
  Saved canonical teacher: checkpoints/teacher_r101.pth

══════════════════════════════════════════════════════
  Teacher training complete. Teacher stats:
    python tools/collect_results.py runs/teachers
  Next: bash tools/run_ablation.sh
══════════════════════════════════════════════════════
[sync] 3 file(s) → Drive  (19:29:35)


In [8]:
# Gate: verify teacher quality before spending grid compute.
# Expect val acc ≳ 95% for all three (may sit 0.1–0.3 pp below v1 — 45k train set).
!python tools/collect_results.py runs/teachers

Teacher | Student   | KD   | alpha | T | Stem  | Seeds       | Val acc (%)  | Test@best (%) | Test@final (%)
--------|-----------|------|-------|---|-------|-------------|--------------|---------------|---------------
-       | resnet101 | none | -     | - | cifar | 3 [0, 1, 2] | 95.95 ± 0.15 | 95.37 ± 0.12  | 95.42 ± 0.29  
-       | resnet34  | none | -     | - | cifar | 3 [0, 1, 2] | 95.95 ± 0.06 | 95.30 ± 0.09  | 95.38 ± 0.03  
-       | resnet50  | none | -     | - | cifar | 3 [0, 1, 2] | 96.07 ± 0.22 | 95.36 ± 0.18  | 95.41 ± 0.30  


## 3 — Stage 1b: Selection grid (48 runs, seed 0, val-based)

4 pairs × (9 Logit-KD + 3 Feature-KD) configs. Test set plays no role here.
Safe to interrupt: re-running the cell resumes where it left off.

In [9]:
!bash tools/run_ablation.sh
sync_to_drive(verbose=True)

  Stage 1: SELECTION grid (val-based, seed 0)
  Pairs: R50->R18 | R34->R18 | R50->R34 | R101->R34

── [1] resnet50_to_resnet18/logit/a0.3_t2/seed0 ──────────────────────────────────
  SKIP: results already exist — runs/select/resnet50_to_resnet18/logit/a0.3_t2/seed0/results.json

── [2] resnet50_to_resnet18/logit/a0.3_t3/seed0 ──────────────────────────────────
  SKIP: results already exist — runs/select/resnet50_to_resnet18/logit/a0.3_t3/seed0/results.json

── [3] resnet50_to_resnet18/logit/a0.3_t4/seed0 ──────────────────────────────────
  SKIP: results already exist — runs/select/resnet50_to_resnet18/logit/a0.3_t4/seed0/results.json

── [4] resnet50_to_resnet18/feature/a0.3/seed0 ──────────────────────────────────
  SKIP: results already exist — runs/select/resnet50_to_resnet18/feature/a0.3/seed0/results.json

── [5] resnet50_to_resnet18/logit/a0.5_t2/seed0 ──────────────────────────────────
  SKIP: results already exist — runs/select/resnet50_to_resnet18/logit/a0.5_t2/seed0/results

## 4 — Stage 1c: Select best configs by val accuracy

In [10]:
!python tools/collect_results.py runs/select --write-best best_configs.json
sync_to_drive(verbose=True)

Teacher   | Student  | KD      | alpha | T | Stem  | Seeds | Val acc (%) | Test@best (%) | Test@final (%)
----------|----------|---------|-------|---|-------|-------|-------------|---------------|---------------
resnet101 | resnet34 | feature | 0.3   | - | cifar | 1 [0] | 95.84       | 95.44         | 95.36         
resnet101 | resnet34 | feature | 0.5   | - | cifar | 1 [0] | 95.68       | 94.95         | 95.08         
resnet101 | resnet34 | feature | 0.7   | - | cifar | 1 [0] | 95.56       | 95.29         | 95.26         
resnet101 | resnet34 | logit   | 0.3   | 2 | cifar | 1 [0] | 95.62       | 95.01         | 95.01         
resnet101 | resnet34 | logit   | 0.3   | 3 | cifar | 1 [0] | 95.50       | 94.94         | 95.07         
resnet101 | resnet34 | logit   | 0.3   | 4 | cifar | 1 [0] | 95.24       | 94.46         | 94.94         
resnet101 | resnet34 | logit   | 0.5   | 2 | cifar | 1 [0] | 95.20       | 94.85         | 94.90         
resnet101 | resnet34 | logit   | 0.5   | 3 | c

## 5 — Stage 2: Final runs (5 seeds) + fidelity

Baselines (R18, R34 × 5 seeds) + every best config × 5 seeds.
`tools/eval.py` runs automatically after each KD run (agreement, KL, per-class acc).
These are the numbers that go in the paper: `test_acc_best`, mean ± std.

In [11]:
!bash tools/run_final.sh
sync_to_drive(verbose=True)

SKIP baseline resnet18 seed0
SKIP baseline resnet18 seed1
SKIP baseline resnet18 seed2
SKIP baseline resnet18 seed3
SKIP baseline resnet18 seed4
SKIP baseline resnet34 seed0
SKIP baseline resnet34 seed1
SKIP baseline resnet34 seed2
SKIP baseline resnet34 seed3
SKIP baseline resnet34 seed4
SKIP resnet101_to_resnet34_feature seed0
SKIP resnet101_to_resnet34_feature seed1
SKIP resnet101_to_resnet34_feature seed2
SKIP resnet101_to_resnet34_feature seed3
SKIP resnet101_to_resnet34_feature seed4
SKIP resnet101_to_resnet34_logit seed0
SKIP resnet101_to_resnet34_logit seed1
SKIP resnet101_to_resnet34_logit seed2
SKIP resnet101_to_resnet34_logit seed3
SKIP resnet101_to_resnet34_logit seed4
SKIP resnet34_to_resnet18_feature seed0
SKIP resnet34_to_resnet18_feature seed1
SKIP resnet34_to_resnet18_feature seed2
SKIP resnet34_to_resnet18_feature seed3
SKIP resnet34_to_resnet18_feature seed4
SKIP resnet34_to_resnet18_logit seed0
SKIP resnet34_to_resnet18_logit seed1
SKIP resnet34_to_resnet18_logit se

In [12]:
!python tools/collect_results.py runs/final --csv final_results.csv

Teacher   | Student  | KD      | alpha | T | Stem  | Seeds             | Val acc (%)  | Test@best (%) | Test@final (%)
----------|----------|---------|-------|---|-------|-------------------|--------------|---------------|---------------
-         | resnet18 | none    | -     | - | cifar | 5 [0, 1, 2, 3, 4] | 95.37 ± 0.18 | 94.86 ± 0.14  | 94.82 ± 0.16  
-         | resnet34 | none    | -     | - | cifar | 5 [0, 1, 2, 3, 4] | 95.58 ± 0.26 | 95.04 ± 0.13  | 95.04 ± 0.12  
resnet101 | resnet34 | feature | 0.3   | - | cifar | 5 [0, 1, 2, 3, 4] | 95.63 ± 0.21 | 95.26 ± 0.11  | 95.23 ± 0.10  
resnet101 | resnet34 | logit   | 0.7   | 2 | cifar | 5 [0, 1, 2, 3, 4] | 95.61 ± 0.13 | 95.10 ± 0.16  | 95.12 ± 0.11  
resnet34  | resnet18 | feature | 0.7   | - | cifar | 5 [0, 1, 2, 3, 4] | 95.59 ± 0.14 | 94.98 ± 0.15  | 95.02 ± 0.08  
resnet34  | resnet18 | logit   | 0.7   | 2 | cifar | 5 [0, 1, 2, 3, 4] | 95.40 ± 0.19 | 94.78 ± 0.09  | 94.80 ± 0.16  
resnet50  | resnet18 | feature | 0.7   | - | cif

In [13]:
# Backfill any fidelity.json lost to a session death between train and eval.
import json, subprocess
from pathlib import Path

best = json.load(open('best_configs.json'))
ckpt_for = {'resnet50': 'checkpoints/teacher_r50.pth',
            'resnet34': 'checkpoints/teacher_r34.pth',
            'resnet101': 'checkpoints/teacher_r101.pth'}
missing = 0
for name, b in best.items():
    for seed in range(5):
        out = Path(f"runs/final/{name.replace('/', '_')}/seed{seed}")
        if (out / 'results.json').exists() and not (out / 'fidelity.json').exists():
            student_ckpt = out / 'checkpoint_best.pth'
            if not student_ckpt.exists():
                print(f'  {out}: fidelity missing AND checkpoint gone — re-run this seed '
                      f'(delete its results.json, then re-run Stage 2).')
                missing += 1
                continue
            print(f'  Backfilling fidelity: {out}')
            subprocess.run(['python', 'tools/eval.py',
                            '--student-arch', b['student'], '--student-weights', str(student_ckpt),
                            '--teacher-arch', b['teacher'], '--teacher-weights', ckpt_for[b['teacher']],
                            '--output', str(out / 'fidelity.json')], check=True)
print('Fidelity check complete.' + (f'  ({missing} unrecoverable — see above)' if missing else ''))
sync_to_drive()

Fidelity check complete.
[sync] 5 file(s) → Drive  (19:30:07)


## 5.5 — Stage 2b: Bug ablation (bugged vs corrected Feature-KD)

Reproduces the v1 gradient-clipping bug (projections excluded from clipping)
under conditions identical to the corrected Stage-2 runs: R50→R18 Feature-KD,
same $\alpha$ (read from `best_configs.json`), seeds {0..4}, same split and
baseline. The corrected counterpart already exists in `runs/final/`.
Records pre-clip gradient norms per epoch → evidence for the paper's
instability claim. Prints max unclipped projection norms at the end — note
these down for `tab:bug`.

In [14]:
!bash tools/run_bug_ablation.sh
sync_to_drive(verbose=True)

Bugged Feature-KD ablation: R50->R18, alpha=0.7, seeds 0 1 2 3 4
SKIP seed0
SKIP seed1
SKIP seed2
SKIP seed3
SKIP seed4
Done. Compare with:
  python tools/collect_results.py runs  # bugged rows show kd=feature(bugged)
Max unclipped projection grad norm per seed:
  seed0: 0.21
  seed1: 0.20
  seed2: 0.21
  seed3: 0.20
  seed4: 0.20
[sync] 5 file(s) → Drive  (19:30:14)


In [15]:
!python tools/collect_results.py runs/bug_ablation

Teacher  | Student  | KD              | alpha | T | Stem  | Seeds             | Val acc (%)  | Test@best (%) | Test@final (%)
---------|----------|-----------------|-------|---|-------|-------------------|--------------|---------------|---------------
resnet50 | resnet18 | feature(bugged) | 0.7   | - | cifar | 5 [0, 1, 2, 3, 4] | 95.40 ± 0.20 | 95.00 ± 0.18  | 94.99 ± 0.18  


## 6 — Stage 3: Stem ablation (ImageNet vs CIFAR stem)

In [ ]:
!bash tools/run_stem_ablation.sh
sync_to_drive(verbose=True)
!python tools/collect_results.py runs/stem_ablation

── resnet18 stem=cifar seed0 ──
2026-07-11 19:31:19.466568: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-11 19:31:19.538521: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
19:31:23 [INFO] train: Seed: 0  |  Split seed (fixed): 1234
19:31:23 [INFO] train: Device: cuda
19:31:23 [INFO] train: GPU: NVIDIA A100-SXM4-80GB
19:31:27 [INFO] train: Train: 45,000  Val: 5,000  Test: 10,000
19:31:27 [INFO] train: Student (resnet18, stem=cifar): 11,173,962 params
19:31:28 [INFO] src.trainer: Starting training for 100 epoch

## 7 — Aggregate + download

Zips **metrics only** (`results.json`, `fidelity.json`, CSV, `best_configs.json`) —
a few MB, not the multi-GB checkpoint archive. Upload this zip to the chat for
Phase 3 (paper revision).

In [ ]:
!python tools/collect_results.py runs/final --csv final_results.csv

Teacher   | Student  | KD      | alpha | T | Stem  | Seeds             | Val acc (%)  | Test@best (%) | Test@final (%)
----------|----------|---------|-------|---|-------|-------------------|--------------|---------------|---------------
-         | resnet18 | none    | -     | - | cifar | 5 [0, 1, 2, 3, 4] | 95.37 ± 0.18 | 94.86 ± 0.14  | 94.82 ± 0.16  
-         | resnet34 | none    | -     | - | cifar | 5 [0, 1, 2, 3, 4] | 95.58 ± 0.26 | 95.04 ± 0.13  | 95.04 ± 0.12  
resnet101 | resnet34 | feature | 0.3   | - | cifar | 5 [0, 1, 2, 3, 4] | 95.63 ± 0.21 | 95.26 ± 0.11  | 95.23 ± 0.10  
resnet101 | resnet34 | logit   | 0.7   | 2 | cifar | 5 [0, 1, 2, 3, 4] | 95.61 ± 0.13 | 95.10 ± 0.16  | 95.12 ± 0.11  
resnet34  | resnet18 | feature | 0.7   | - | cifar | 5 [0, 1, 2, 3, 4] | 95.59 ± 0.14 | 94.98 ± 0.15  | 95.02 ± 0.08  
resnet34  | resnet18 | logit   | 0.7   | 2 | cifar | 5 [0, 1, 2, 3, 4] | 95.40 ± 0.19 | 94.78 ± 0.09  | 94.80 ± 0.16  
resnet50  | resnet18 | feature | 0.7   | - | cif

In [ ]:
import subprocess
from pathlib import Path

files = [str(p) for p in Path('runs').rglob('results.json')]
files += [str(p) for p in Path('runs').rglob('fidelity.json')]
files += [f for f in ['final_results.csv', 'best_configs.json'] if Path(f).exists()]

archive = '/content/kd_v2_results.zip'
subprocess.run(['zip', '-q', archive] + files, check=True)
print(f'{archive}: {Path(archive).stat().st_size/1e6:.1f} MB, {len(files)} files')

shutil.copy2(archive, DRIVE_BASE / 'kd_v2_results.zip')
print('Also saved to Drive.')

from google.colab import files as colab_files
colab_files.download(archive)

/content/kd_v2_results.zip: 0.8 MB, 168 files
Also saved to Drive.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Session-resume cheat sheet

1. New runtime → run the setup block top-to-bottom: GPU check → Drive mount →
   clone/pull + restore → pip → CIFAR Drive-cache (→ fetch only if cache
   missed).
2. Jump to the stage cell that was interrupted and re-run it. Everything
   already completed SKIPs automatically.
3. Order of remaining work: Stage 2 → backfill → Stage 2b (bug ablation) →
   Stage 3 (stem) → Stage 4 (aggregate + zip). Stages 2b and 3 are
   independent of each other.

### Optional: feat_norm ablation
Ask before running `FEAT_NORM=teacher_std` — it needs a separate output dir
to avoid colliding with `runs/select`.